In [ ]:
import medmnist
from medmnist import PathMNIST
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
data_dir = '/path/to/thesis_project/data/raw/path_mnist'

ds_train = PathMNIST(split='train', download=True, root=data_dir)
ds_val   = PathMNIST(split='val',   download=True, root=data_dir)
ds_test  = PathMNIST(split='test',  download=True, root=data_dir)

In [ ]:
#dataset basic info

print(ds_train, "\n")
print(ds_train.info, "\n")
print(type(ds_train.info), "\n")
print(ds_train.info.keys(), "\n")

class_names = ds_train.info["label"]
num_classes = len(class_names)

print("Number of training samples:", len(ds_train))
print("Number of validation samples:", len(ds_val))
print("Number of test samples:", len(ds_test))
print("Number of classes:", num_classes)
print("Class names:", class_names)

image, label = ds_train[0]
image_np = np.array(image)
#inspecting data format quickly
print("image shape:", image_np.shape)
print("image dtype:", image_np.dtype)
print("pixel range: [{}, {}]".format(image_np.min(), image_np.max() ))

In [ ]:
# pixel range check
pixel_min = np.inf
pixel_max = -np.inf

for split in [ds_train, ds_val, ds_test]:
    for image, _ in split:
        img = np.array(image)
        pixel_min = min(pixel_min, img.min())
        pixel_max = max(pixel_max, img.max())

print("dtype:", np.array(ds_train[0][0]).dtype)
print("pixel min:", int(pixel_min))
print("pixel max:", int(pixel_max))

In [ ]:
#class distribution - samples per class (train)

counts = np.zeros(num_classes, dtype = int)
for _, label in ds_train:
    counts[int(label[0])] += 1

#plot

plt.figure( figsize = (10, 4) )
plt.bar(list(class_names.values()), counts)
plt.title("path_mnist class distribution (train)")
plt.xlabel("Class")
plt.ylabel("Number of samples")
plt.xticks(rotation = 45, ha = "right")
plt.tight_layout()
plt.savefig("/path/to/thesis_project/eda/path_mnist_class_distribution_(train).png",
            dpi = 300, bbox_inches = "tight")
plt.show()

for name, count in zip(class_names.values(), counts):
    print("{}: {}".format(name, count))

In [ ]:
#sample images per class
n_per_class = 5
samples = {i: [] for i in range(num_classes)}
for image, label in ds_train:
    l = int(label[0])
    if len(samples[l]) < n_per_class:
        samples[l].append(np.array(image))
    if all(len(v) == n_per_class for v in samples.values()):
        break

# plot
fig, axes = plt.subplots(num_classes, n_per_class, figsize=(10, 20))
for i in range(num_classes):
    for j in range(n_per_class):
        axes[i, j].imshow(samples[i][j])
        axes[i, j].axis("off")

plt.tight_layout()
plt.savefig("/path/to/thesis_project/eda/path_mnist_sample_images_per_class.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#pixel statistics per channel

channel_sum = np.zeros(3)
channel_sq_sum = np.zeros(3)
n_pixels = 0

for image, _ in ds_train:
    img = np.array(image).astype(np.float32) / 255.0
    channel_sum += img.reshape(-1, 3).sum(axis = 0)
    channel_sq_sum += (img.reshape(-1,3)**2).sum(axis = 0)
    n_pixels += img.shape[0]*img.shape[1]

mean = channel_sum / n_pixels
std = np.sqrt(channel_sq_sum / n_pixels - mean **2)

for i, channel in enumerate(["R","G","B"]):
    print("Channel {}: mean = {:.4f}, std = {:.4f}".format(channel, mean[i], std[i]))

#plot

all_pixels = {0: [], 1: [], 2: []}

for image, _ in ds_train:
    img = np.array(image).astype(np.float32) / 255.0
    for c in range(3):
        all_pixels[c].append(img[:,:,c].flatten())

for c in range(3):
    all_pixels[c] = np.concatenate(all_pixels[c])

colors = ["red", "green", "blue"]
channel_names = ["R","G","B"]

plt.figure(figsize = (10,4))
for c in range(3):
    plt.hist(all_pixels[c], bins = 50, alpha = 0.5, color = colors[c], label = channel_names[c])

plt.title("path_mnist pixel statistics per channel (train)")
plt.xlabel("Pixel value")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.savefig("/path/to/thesis_project/eda/path_mnist_pixel_statistics_per_channel.png",
            dpi = 300, bbox_inches = "tight")
plt.show()